# 01 — Binary Temporal Leakage AuditRandom Forest, LightGBM, XGBoost, and MLP evaluated under random vs. temporal train-test splits on CIC-IDS 2017 (binary BENIGN/ATTACK formulation). Produces the results in Table 2 of the paper.**Assumes** `day_dfs` preprocessed CSVs are available at `SAVE_PATH` (see README for preprocessing details).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import (accuracy_score, f1_score,
                              confusion_matrix, roc_auc_score)
from imblearn.over_sampling import SMOTE
from collections import Counter
import gc, os, warnings
warnings.filterwarnings('ignore')

SAVE_PATH    = '/content/drive/MyDrive/New Approach/cicids2017-01/cicids2017/preprocessed'
LABEL_COL    = 'Label'
RANDOM_STATE = 42
ordered_days = ['monday','tuesday','wednesday','thursday','friday']

# ── reload day_dfs if needed ──
if 'day_dfs' not in dir() or len(day_dfs) == 0:
    day_dfs = {}
    for day in ordered_days:
        path = SAVE_PATH + f'/{day}_preprocessed.csv'
        if not os.path.exists(path): continue
        df        = pd.read_csv(path, low_memory=False)
        df['day'] = day
        day_dfs[day] = df
        print(f'{day:12s}: {len(df):>8,} rows')
    print(f'Total: {sum(len(v) for v in day_dfs.values()):,}')


# ══════════════════════════════════════════════════════════════
#  PREPROCESSING
# ══════════════════════════════════════════════════════════════
def preprocess_split(df_tr, df_te, label_col=LABEL_COL,
                     variance_thresh=0.01, corr_thresh=0.95):
    try:
        X_tr     = df_tr.drop(columns=[label_col,'day'], errors='ignore').copy()
        X_te     = df_te.drop(columns=[label_col,'day'], errors='ignore').copy()
        y_tr_raw = df_tr[label_col].copy()
        y_te_raw = df_te[label_col].copy()

        X_tr.drop(columns=X_tr.select_dtypes(exclude=[np.number]).columns, inplace=True)
        X_te.drop(columns=X_te.select_dtypes(exclude=[np.number]).columns, inplace=True)
        X_tr.replace([np.inf,-np.inf], np.nan, inplace=True)
        X_te.replace([np.inf,-np.inf], np.nan, inplace=True)
        mask_tr  = X_tr.notna().all(axis=1)
        mask_te  = X_te.notna().all(axis=1)
        X_tr     = X_tr[mask_tr].reset_index(drop=True)
        y_tr_raw = y_tr_raw[mask_tr].reset_index(drop=True)
        X_te     = X_te[mask_te].reset_index(drop=True)
        y_te_raw = y_te_raw[mask_te].reset_index(drop=True)

        common = [c for c in X_tr.columns if c in X_te.columns]
        X_tr = X_tr[common].copy(); X_te = X_te[common].copy()

        vt   = VarianceThreshold(threshold=variance_thresh)
        arr  = vt.fit_transform(X_tr)
        cols = np.array(common)[vt.get_support()]
        X_tr = pd.DataFrame(arr, columns=cols)
        X_te = pd.DataFrame(vt.transform(X_te), columns=cols)

        corr  = X_tr.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        drop  = [c for c in upper.columns if any(upper[c] >= corr_thresh)]
        X_tr.drop(columns=drop, inplace=True)
        X_te.drop(columns=drop, inplace=True, errors='ignore')
        feature_names = X_tr.columns.tolist()
        print(f'  Features: {len(feature_names)}')

        scaler  = StandardScaler()
        X_tr_sc = pd.DataFrame(scaler.fit_transform(X_tr), columns=feature_names)
        X_te_sc = pd.DataFrame(scaler.transform(X_te),     columns=feature_names)

        le = LabelEncoder()
        le.fit(['BENIGN','ATTACK'])
        y_tr_raw = y_tr_raw.where(y_tr_raw.isin(['BENIGN','ATTACK']), 'ATTACK')
        y_te_raw = y_te_raw.where(y_te_raw.isin(['BENIGN','ATTACK']), 'ATTACK')
        y_tr = pd.Series(le.transform(y_tr_raw), name='label')
        y_te = pd.Series(le.transform(y_te_raw), name='label')

        counts  = Counter(y_tr)
        min_cls = min(counts, key=counts.get)
        maj_cls = max(counts, key=counts.get)
        min_cnt = counts[min_cls]
        maj_cnt = counts[maj_cls]
        k       = max(1, min(5, min_cnt - 1))
        target  = max(min_cnt, min(30_000, maj_cnt))
        strat   = {min_cls: target}

        print(f'  SMOTE k={k} | {min_cnt}→{target} | '
              f'Train dist: {dict(counts)} | '
              f'BENIGN={le.classes_.tolist().index("BENIGN")} '
              f'ATTACK={le.classes_.tolist().index("ATTACK")}')
        smote = SMOTE(random_state=RANDOM_STATE,
                      k_neighbors=k, sampling_strategy=strat)
        X_sm, y_sm = smote.fit_resample(X_tr_sc, y_tr)
        print(f'  After SMOTE: {len(y_sm):,} | Test: {len(y_te):,}')
        return X_sm, y_sm, X_te_sc, y_te, le, feature_names

    except Exception as e:
        import traceback
        print(f'  [ERROR]: {e}')
        traceback.print_exc()
        return None, None, None, None, None, None


# ══════════════════════════════════════════════════════════════
#  EVALUATE — FPR FIX APPLIED HERE
# ══════════════════════════════════════════════════════════════
def evaluate_model(model, X_te, y_te, model_name, split_name, le):
    """
    Compute all metrics. Dynamically finds BENIGN/ATTACK class index
    from le instead of assuming BENIGN=0 — fixes FPR computation
    when LabelEncoder assigns BENIGN=1, ATTACK=0 (as happens in
    the temporal split).
    """
    preds = model.predict(X_te)

    acc = accuracy_score(y_te, preds)
    f1m = f1_score(y_te, preds, average='macro',    zero_division=0)
    f1w = f1_score(y_te, preds, average='weighted', zero_division=0)
    f1b = f1_score(y_te, preds, average='binary',   zero_division=0)

    classes    = list(le.classes_)
    benign_idx = classes.index('BENIGN')
    attack_idx = classes.index('ATTACK')

    atk_mask   = (y_te == attack_idx)
    atk_recall = (preds[atk_mask] == attack_idx).mean() \
                 if atk_mask.any() else 0.0

    cm  = confusion_matrix(y_te, preds, labels=[benign_idx, attack_idx])
    TN  = cm[0, 0]
    FP  = cm[0, 1]
    FPR = FP / (FP + TN) if (FP + TN) > 0 else 0.0

    try:
        proba = model.predict_proba(X_te)[:, attack_idx]
        roc   = roc_auc_score((y_te == attack_idx).astype(int), proba)
    except Exception:
        roc = float('nan')

    print(f'  [{split_name:8s}] {model_name:14s} | '
          f'Acc:{acc:.4f} | F1m:{f1m:.4f} | '
          f'Recall:{atk_recall:.4f} | FPR:{FPR:.4f} | '
          f'AUC:{roc:.4f}  [BENIGN={benign_idx} ATTACK={attack_idx}]')

    return {'model':model_name, 'split':split_name,
            'accuracy':round(acc,4), 'f1_weighted':round(f1w,4),
            'f1_macro':round(f1m,4), 'f1_binary':round(f1b,4),
            'attack_recall':round(atk_recall,4),
            'FPR':round(FPR,4),
            'ROC_AUC':round(roc,4) if not np.isnan(roc) else None}


# ══════════════════════════════════════════════════════════════
#  BUILD SPLITS
# ══════════════════════════════════════════════════════════════

# ── TEMPORAL split ──
print('\nBuilding TEMPORAL split...')
df_tr_raw = pd.concat(
    [day_dfs[d] for d in ['monday','tuesday','wednesday','thursday']],
    ignore_index=True)
df_te_raw = day_dfs['friday'].copy()

for df in [df_tr_raw, df_te_raw]:
    df[LABEL_COL] = df[LABEL_COL].apply(
        lambda x: 'BENIGN' if x=='BENIGN' else 'ATTACK')

if len(df_tr_raw) > 200_000:
    df_tr_raw = (df_tr_raw.groupby(LABEL_COL, group_keys=False)
                 .apply(lambda x: x.sample(
                     min(len(x), int(200_000*len(x)/len(df_tr_raw))),
                     random_state=RANDOM_STATE))
                 .reset_index(drop=True))
if len(df_te_raw) > 100_000:
    df_te_raw = (df_te_raw.groupby(LABEL_COL, group_keys=False)
                 .apply(lambda x: x.sample(
                     min(len(x), int(100_000*len(x)/len(df_te_raw))),
                     random_state=RANDOM_STATE))
                 .reset_index(drop=True))

print(f'  Train:{len(df_tr_raw):,} dist:{df_tr_raw[LABEL_COL].value_counts().to_dict()}')
print(f'  Test :{len(df_te_raw):,} dist:{df_te_raw[LABEL_COL].value_counts().to_dict()}')

result = preprocess_split(df_tr_raw, df_te_raw)
assert result[0] is not None, "Temporal split failed"
X_tr_temp, y_tr_temp, X_te_temp, y_te_temp, le_temp, feats_temp = result
del df_tr_raw, df_te_raw; gc.collect()
print('Temporal split ready ✓')


# ── RANDOM split ──
print('\nBuilding RANDOM split...')
df_all = pd.concat(day_dfs.values(), ignore_index=True)
df_all[LABEL_COL] = df_all[LABEL_COL].apply(
    lambda x: 'BENIGN' if x=='BENIGN' else 'ATTACK')

if len(df_all) > 200_000:
    df_all = (df_all.groupby(LABEL_COL, group_keys=False)
              .apply(lambda x: x.sample(
                  min(len(x), int(200_000*len(x)/len(df_all))),
                  random_state=RANDOM_STATE))
              .reset_index(drop=True))

X_all = df_all.drop(columns=[LABEL_COL,'day'], errors='ignore')
y_all = df_all[LABEL_COL]
X_r_tr, X_r_te, y_r_tr, y_r_te = train_test_split(
    X_all, y_all, test_size=0.2,
    random_state=RANDOM_STATE, stratify=y_all)
df_r_tr = pd.concat([X_r_tr, y_r_tr], axis=1)
df_r_te = pd.concat([X_r_te, y_r_te], axis=1)

print(f'  Train:{len(df_r_tr):,} | Test:{len(df_r_te):,}')
result = preprocess_split(df_r_tr, df_r_te)
assert result[0] is not None, "Random split failed"
X_tr_rand, y_tr_rand, X_te_rand, y_te_rand, le_rand, feats_rand = result
del df_all, df_r_tr, df_r_te, X_all, y_all; gc.collect()
print('Random split ready ✓')


# ══════════════════════════════════════════════════════════════
#  RUN AUDIT
# ══════════════════════════════════════════════════════════════
models = {
    'RandomForest': RandomForestClassifier(
        n_estimators=100, max_depth=10,
        class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=100, max_depth=8, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8,
        class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=100, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0),
}

results = []
print('\n' + '='*70)
print('MULTI-MODEL INFLATION AUDIT (FPR FIXED)')
print('='*70)

for model_name, model in models.items():
    print(f'\n--- {model_name} ---')

    model.fit(X_tr_rand, y_tr_rand)
    results.append(evaluate_model(
        model, X_te_rand, y_te_rand, model_name, 'Random', le_rand))
    gc.collect()

    model.fit(X_tr_temp, y_tr_temp)
    results.append(evaluate_model(
        model, X_te_temp, y_te_temp, model_name, 'Temporal', le_temp))
    gc.collect()


# ══════════════════════════════════════════════════════════════
#  PAPER 1 TABLE
# ══════════════════════════════════════════════════════════════
df_r = pd.DataFrame(results)

print('\n\n' + '='*70)
print('TABLE 1 — TEMPORAL LEAKAGE BIAS: THREE MODELS × TWO PROTOCOLS')
print('='*70)
print(f'\n{"Model":14s} {"Split":10s} {"Accuracy":>10} {"F1-macro":>10} '
      f'{"Atk-Recall":>12} {"FPR":>8} {"ROC-AUC":>10}')
print('─'*78)

for mn in ['RandomForest','LightGBM','XGBoost']:
    for sp in ['Random','Temporal']:
        row = df_r[(df_r.model==mn)&(df_r.split==sp)].iloc[0]
        print(f'{mn:14s} {sp:10s} {row.accuracy:>10.4f} '
              f'{row.f1_macro:>10.4f} {row.attack_recall:>12.4f} '
              f'{row.FPR:>8.4f} {str(row.ROC_AUC):>10}')
    print()

print('\nINFLATION (Random − Temporal) — THE CORE FINDING:')
print(f'{"Model":14s} {"Δ Accuracy":>12} {"Δ F1-macro":>12} '
      f'{"Δ Atk-Recall":>14} {"Δ Average":>12}')
print('─'*56)

for mn in ['RandomForest','LightGBM','XGBoost']:
    rand = df_r[(df_r.model==mn)&(df_r.split=='Random')].iloc[0]
    temp = df_r[(df_r.model==mn)&(df_r.split=='Temporal')].iloc[0]
    da   = rand.accuracy      - temp.accuracy
    df1  = rand.f1_macro      - temp.f1_macro
    dr   = rand.attack_recall - temp.attack_recall
    avg  = (da + df1 + dr) / 3
    print(f'{mn:14s} {da:>+12.4f} {df1:>+12.4f} {dr:>+14.4f} {avg:>+12.4f}')

df_r.to_csv(SAVE_PATH + '/multi_model_inflation_audit.csv', index=False)
print(f'\nSaved → multi_model_inflation_audit.csv')
print('\nThis is Table 1 of Paper 1.')

monday      :  529,481 rows
tuesday     :  445,645 rows
wednesday   :  691,406 rows
thursday    :  458,626 rows
friday      :  702,718 rows
Total: 2,827,876

Building TEMPORAL split...
  Train:199,999 dist:{'BENIGN': 174799, 'ATTACK': 25200}
  Test :99,999 dist:{'BENIGN': 58904, 'ATTACK': 41095}
  Features: 44
  SMOTE k=5 | 25200→30000 | Train dist: {0: 25200, 1: 174799} | BENIGN=1 ATTACK=0
  After SMOTE: 204,799 | Test: 99,999
Temporal split ready ✓

Building RANDOM split...
  Train:159,999 | Test:40,000
  Features: 44
  SMOTE k=5 | 31490→31490 | Train dist: {1: 128509, 0: 31490} | BENIGN=1 ATTACK=0
  After SMOTE: 159,999 | Test: 40,000
Random split ready ✓

MULTI-MODEL INFLATION AUDIT (FPR FIXED)

--- RandomForest ---
  [Random  ] RandomForest   | Acc:0.9968 | F1m:0.9950 | Recall:0.9966 | FPR:0.0031 | AUC:0.9998  [BENIGN=1 ATTACK=0]
  [Temporal] RandomForest   | Acc:0.6146 | F1m:0.4399 | Recall:0.0683 | FPR:0.0042 | AUC:0.9083  [BENIGN=1 ATTACK=0]

--- LightGBM ---
  [Random  ] Light

## MLP BaselineExtends the three-model tree-based audit above with a multilayer perceptron baseline, confirming the Temporal Leakage Bias generalises beyond tree ensembles.

In [3]:
import pandas as pd
import numpy as np
import os
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE
from collections import Counter
import gc, warnings
warnings.filterwarnings('ignore')

SAVE_PATH    = '/content/drive/MyDrive/New Approach/cicids2017-01/cicids2017/preprocessed'
LABEL_COL    = 'Label'
RANDOM_STATE = 42
ordered_days = ['monday','tuesday','wednesday','thursday','friday']
CATEGORICAL_EXCLUDE = ['Destination Port', 'Source Port']

if 'day_dfs' not in dir() or len(day_dfs) == 0:
    day_dfs = {}
    for day in ordered_days:
        path = SAVE_PATH + f'/{day}_preprocessed.csv'
        if not os.path.exists(path): continue
        df        = pd.read_csv(path, low_memory=False)
        df['day'] = day
        day_dfs[day] = df
    print(f'Loaded {sum(len(v) for v in day_dfs.values()):,} rows')


# ══════════════════════════════════════════════════════════════
#  PREPROCESSING — same pipeline as the tree-model audit
# ══════════════════════════════════════════════════════════════
def preprocess_binary(df_tr, df_te, label_col=LABEL_COL,
                      variance_thresh=0.01, corr_thresh=0.95):
    try:
        X_tr     = df_tr.drop(columns=[label_col,'day'], errors='ignore').copy()
        X_te     = df_te.drop(columns=[label_col,'day'], errors='ignore').copy()
        y_tr_raw = df_tr[label_col].copy()
        y_te_raw = df_te[label_col].copy()

        port_cols = [c for c in X_tr.columns if c in CATEGORICAL_EXCLUDE]
        X_tr.drop(columns=port_cols, inplace=True, errors='ignore')
        X_te.drop(columns=port_cols, inplace=True, errors='ignore')

        X_tr.drop(columns=X_tr.select_dtypes(exclude=[np.number]).columns, inplace=True)
        X_te.drop(columns=X_te.select_dtypes(exclude=[np.number]).columns, inplace=True)
        X_tr.replace([np.inf,-np.inf], np.nan, inplace=True)
        X_te.replace([np.inf,-np.inf], np.nan, inplace=True)
        mask_tr  = X_tr.notna().all(axis=1)
        mask_te  = X_te.notna().all(axis=1)
        X_tr     = X_tr[mask_tr].reset_index(drop=True)
        y_tr_raw = y_tr_raw[mask_tr].reset_index(drop=True)
        X_te     = X_te[mask_te].reset_index(drop=True)
        y_te_raw = y_te_raw[mask_te].reset_index(drop=True)

        common = [c for c in X_tr.columns if c in X_te.columns]
        X_tr = X_tr[common].copy(); X_te = X_te[common].copy()

        vt   = VarianceThreshold(threshold=variance_thresh)
        arr  = vt.fit_transform(X_tr)
        cols = np.array(common)[vt.get_support()]
        X_tr = pd.DataFrame(arr, columns=cols)
        X_te = pd.DataFrame(vt.transform(X_te), columns=cols)

        corr  = X_tr.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        drop  = [c for c in upper.columns if any(upper[c] >= corr_thresh)]
        X_tr.drop(columns=drop, inplace=True)
        X_te.drop(columns=drop, inplace=True, errors='ignore')
        feature_names = X_tr.columns.tolist()

        scaler  = StandardScaler()
        X_tr_sc = pd.DataFrame(scaler.fit_transform(X_tr), columns=feature_names)
        X_te_sc = pd.DataFrame(scaler.transform(X_te),     columns=feature_names)

        le = LabelEncoder()
        le.fit(['BENIGN','ATTACK'])
        y_tr_raw = y_tr_raw.where(y_tr_raw.isin(['BENIGN','ATTACK']), 'ATTACK')
        y_te_raw = y_te_raw.where(y_te_raw.isin(['BENIGN','ATTACK']), 'ATTACK')
        y_tr = pd.Series(le.transform(y_tr_raw), name='label')
        y_te = pd.Series(le.transform(y_te_raw), name='label')

        counts  = Counter(y_tr)
        min_cls = min(counts, key=counts.get)
        maj_cls = max(counts, key=counts.get)
        min_cnt = counts[min_cls]
        maj_cnt = counts[maj_cls]
        k       = max(1, min(5, min_cnt - 1))
        target  = max(min_cnt, min(30_000, maj_cnt))
        strat   = {min_cls: target}

        smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=k, sampling_strategy=strat)
        X_sm, y_sm = smote.fit_resample(X_tr_sc, y_tr)
        print(f'  Features: {len(feature_names)} | After SMOTE: {len(y_sm):,} | Test: {len(y_te):,}')
        return X_sm, y_sm, X_te_sc, y_te, le

    except Exception as e:
        import traceback
        print(f'  [ERROR]: {e}')
        traceback.print_exc()
        return (None,)*5


def evaluate(model, X_te, y_te, le, model_name, split_name):
    preds = model.predict(X_te)
    classes    = list(le.classes_)
    benign_idx = classes.index('BENIGN')
    attack_idx = classes.index('ATTACK')

    acc = accuracy_score(y_te, preds)
    f1m = f1_score(y_te, preds, average='macro', zero_division=0)
    f1w = f1_score(y_te, preds, average='weighted', zero_division=0)

    atk_mask   = (y_te == attack_idx)
    atk_recall = (preds[atk_mask] == attack_idx).mean() if atk_mask.any() else 0.0

    cm  = confusion_matrix(y_te, preds, labels=[benign_idx, attack_idx])
    TN, FP = cm[0,0], cm[0,1]
    FPR = FP / (FP + TN) if (FP + TN) > 0 else 0.0

    try:
        proba = model.predict_proba(X_te)[:, attack_idx]
        roc   = roc_auc_score((y_te == attack_idx).astype(int), proba)
    except Exception:
        roc = float('nan')

    print(f'  [{split_name:8s}] {model_name:10s} | '
          f'Acc:{acc:.4f} | F1m:{f1m:.4f} | Recall:{atk_recall:.4f} | '
          f'FPR:{FPR:.4f} | AUC:{roc:.4f}')

    return {'model': model_name, 'split': split_name,
            'accuracy': round(acc,4), 'f1_weighted': round(f1w,4),
            'f1_macro': round(f1m,4), 'attack_recall': round(atk_recall,4),
            'FPR': round(FPR,4),
            'ROC_AUC': round(roc,4) if not np.isnan(roc) else None}


# ══════════════════════════════════════════════════════════════
#  BUILD SPLITS (same construction as the tree-model audit)
# ══════════════════════════════════════════════════════════════
print('\nBuilding TEMPORAL split...')
df_tr_raw = pd.concat(
    [day_dfs[d] for d in ['monday','tuesday','wednesday','thursday']],
    ignore_index=True)
df_te_raw = day_dfs['friday'].copy()
for df in [df_tr_raw, df_te_raw]:
    df[LABEL_COL] = df[LABEL_COL].apply(lambda x: 'BENIGN' if x=='BENIGN' else 'ATTACK')

if len(df_tr_raw) > 200_000:
    df_tr_raw = (df_tr_raw.groupby(LABEL_COL, group_keys=False)
                 .apply(lambda x: x.sample(
                     min(len(x), int(200_000*len(x)/len(df_tr_raw))),
                     random_state=RANDOM_STATE))
                 .reset_index(drop=True))
if len(df_te_raw) > 100_000:
    df_te_raw = (df_te_raw.groupby(LABEL_COL, group_keys=False)
                 .apply(lambda x: x.sample(
                     min(len(x), int(100_000*len(x)/len(df_te_raw))),
                     random_state=RANDOM_STATE))
                 .reset_index(drop=True))

result = preprocess_binary(df_tr_raw, df_te_raw)
assert result[0] is not None, "Temporal split failed"
X_tr_temp, y_tr_temp, X_te_temp, y_te_temp, le_temp = result
del df_tr_raw, df_te_raw; gc.collect()
print('Temporal split ready ✓')

print('\nBuilding RANDOM split...')
df_all = pd.concat(day_dfs.values(), ignore_index=True)
df_all[LABEL_COL] = df_all[LABEL_COL].apply(lambda x: 'BENIGN' if x=='BENIGN' else 'ATTACK')
if len(df_all) > 200_000:
    df_all = (df_all.groupby(LABEL_COL, group_keys=False)
              .apply(lambda x: x.sample(
                  min(len(x), int(200_000*len(x)/len(df_all))),
                  random_state=RANDOM_STATE))
              .reset_index(drop=True))
X_all = df_all.drop(columns=[LABEL_COL,'day'], errors='ignore')
y_all = df_all[LABEL_COL]
X_r_tr, X_r_te, y_r_tr, y_r_te = train_test_split(
    X_all, y_all, test_size=0.2, random_state=RANDOM_STATE, stratify=y_all)
df_r_tr = pd.concat([X_r_tr, y_r_tr], axis=1)
df_r_te = pd.concat([X_r_te, y_r_te], axis=1)

result = preprocess_binary(df_r_tr, df_r_te)
assert result[0] is not None, "Random split failed"
X_tr_rand, y_tr_rand, X_te_rand, y_te_rand, le_rand = result
del df_all, df_r_tr, df_r_te, X_all, y_all; gc.collect()
print('Random split ready ✓')


# ══════════════════════════════════════════════════════════════
#  MLP BASELINE
# ══════════════════════════════════════════════════════════════
print('\n' + '='*70)
print('MLP DEEP LEARNING BASELINE')
print('='*70)

def make_mlp(seed=RANDOM_STATE):
    # modest architecture deliberately — reviewer asked for "one
    # deep model", not an exhaustive architecture search
    return MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        solver='adam',
        alpha=1e-4,                  # L2 regularisation
        batch_size=256,
        learning_rate_init=1e-3,
        max_iter=100,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
        random_state=seed,
    )

mlp_results = []

print('\n--- MLP on Random split ---')
mlp_rand = make_mlp()
mlp_rand.fit(X_tr_rand, y_tr_rand)
mlp_results.append(evaluate(mlp_rand, X_te_rand, y_te_rand, le_rand, 'MLP', 'Random'))
print(f'  Converged in {mlp_rand.n_iter_} iterations '
      f'(stopped early: {mlp_rand.n_iter_ < 100})')
gc.collect()

print('\n--- MLP on Temporal split ---')
mlp_temp = make_mlp()
mlp_temp.fit(X_tr_temp, y_tr_temp)
mlp_results.append(evaluate(mlp_temp, X_te_temp, y_te_temp, le_temp, 'MLP', 'Temporal'))
print(f'  Converged in {mlp_temp.n_iter_} iterations '
      f'(stopped early: {mlp_temp.n_iter_ < 100})')
gc.collect()


# ══════════════════════════════════════════════════════════════
#  RESULTS — ready to merge with the tree-model table
# ══════════════════════════════════════════════════════════════
df_mlp = pd.DataFrame(mlp_results)
print('\n' + '='*70)
print('MLP RESULTS — APPEND TO TABLE 2')
print('='*70)
print(df_mlp.to_string(index=False))

rand_row = df_mlp[df_mlp.split=='Random'].iloc[0]
temp_row = df_mlp[df_mlp.split=='Temporal'].iloc[0]
print('\nMLP inflation (Random − Temporal):')
print(f'  Δ Accuracy:    {rand_row.accuracy - temp_row.accuracy:+.4f} '
      f'({(rand_row.accuracy - temp_row.accuracy)*100:+.2f} pp)')
print(f'  Δ F1-Macro:    {rand_row.f1_macro - temp_row.f1_macro:+.4f} '
      f'({(rand_row.f1_macro - temp_row.f1_macro)*100:+.2f} pp)')
print(f'  Δ Atk-Recall:  {rand_row.attack_recall - temp_row.attack_recall:+.4f} '
      f'({(rand_row.attack_recall - temp_row.attack_recall)*100:+.2f} pp)')

df_mlp.to_csv(SAVE_PATH + '/mlp_baseline_audit.csv', index=False)
print(f'\nSaved → mlp_baseline_audit.csv')
print('\nThis row can be appended directly to Table 2 (tree-model audit)')
print('to show the bias generalises beyond tree-based models.')


Building TEMPORAL split...
  Features: 43 | After SMOTE: 204,799 | Test: 99,999
Temporal split ready ✓

Building RANDOM split...
  Features: 43 | After SMOTE: 159,999 | Test: 40,000
Random split ready ✓

MLP DEEP LEARNING BASELINE

--- MLP on Random split ---
  [Random  ] MLP        | Acc:0.9722 | F1m:0.9552 | Recall:0.9047 | FPR:0.0113 | AUC:0.9954
  Converged in 25 iterations (stopped early: True)

--- MLP on Temporal split ---
  [Temporal] MLP        | Acc:0.6977 | F1m:0.6123 | Recall:0.2778 | FPR:0.0093 | AUC:0.8783
  Converged in 47 iterations (stopped early: True)

MLP RESULTS — APPEND TO TABLE 2
model    split  accuracy  f1_weighted  f1_macro  attack_recall    FPR  ROC_AUC
  MLP   Random    0.9722       0.9719    0.9552         0.9047 0.0113   0.9954
  MLP Temporal    0.6977       0.6447    0.6123         0.2778 0.0093   0.8783

MLP inflation (Random − Temporal):
  Δ Accuracy:    +0.2745 (+27.45 pp)
  Δ F1-Macro:    +0.3429 (+34.29 pp)
  Δ Atk-Recall:  +0.6269 (+62.69 pp)

Save